# Python *args and **kwargs: From Zero to Practical Use

You will often see function signatures like:

```python
def wrapper(*args, **kwargs):
    ...
```

This notebook teaches that syntax step by step:

1. Why variable arguments exist
2. How `*args` collects positional values
3. How `**kwargs` collects keyword values
4. The correct parameter order in a function definition
5. Real patterns you will see in production code

No external libraries required — only the Python standard library.

### Before you run any cell

Select this kernel in the notebook toolbar (top-right):

**`Python (langgraph-training)`**

That kernel uses this project's `.venv` where `ipykernel` is installed.  
If cells keep spinning with no output, the wrong Python interpreter is selected.

In [ ]:
import sys

print("Kernel OK")
print(sys.executable)

## Part 0 — Why this syntax exists

A normal function expects a **fixed** number of arguments:

```python
def add(a, b):
    return a + b
```

But real code often needs to accept **any number** of extra inputs:

- A logger that records arbitrary metadata
- A wrapper that forwards arguments to another function
- An API helper that accepts optional query parameters

### Naming note

`args` and `kwargs` are **conventions**, not Python keywords.  
The operators that matter are `*` and `**`:

| Syntax | In a function **definition** | Result type |
|--------|------------------------------|-------------|
| `*args` | Collect extra **positional** arguments | `tuple` |
| `**kwargs` | Collect extra **keyword** arguments | `dict` |

## Part 1 — `*args` (positional varargs)

`*args` gathers all extra positional arguments into a **tuple**.

In [ ]:
def total(*args):
    """Sum any number of numeric positional arguments."""
    print(f"Received args: {args} (type: {type(args).__name__})")
    return sum(args)


print(total(1, 2))
print(total(10, 20, 30, 40))

In [ ]:
def greet(greeting, *names):
    """First argument is required; the rest are collected in *names."""
    for name in names:
        print(f"{greeting}, {name}!")


greet("Hello", "Alex", "Sam", "Lee")

## Part 2 — `**kwargs` (keyword varargs)

`**kwargs` gathers all extra keyword arguments into a **dictionary**.

In [ ]:
def log_event(event_name, **kwargs):
    """Log an event plus arbitrary keyword metadata."""
    print(f"Event: {event_name}")
    print(f"Metadata: {kwargs} (type: {type(kwargs).__name__})")
    for key, value in kwargs.items():
        print(f"  {key} = {value!r}")


log_event("login", user="alex", ip="127.0.0.1", success=True)

## Part 3 — Full parameter order (critical rule)

When you combine all parameter kinds, Python enforces this order:

```python
def f(pos1, pos2, *args, kw_only=default, **kwargs):
    ...
```

| Parameter kind | Example | Collected as |
|----------------|---------|--------------|
| Positional | `a, b` | normal binding |
| `*args` | extra positionals | `tuple` |
| Keyword-only | `limit` after `*` | must be passed by name |
| `**kwargs` | extra keywords | `dict` |

In [ ]:
def inspect_call(a, b, *args, limit=10, **kwargs):
    """Show where each argument lands."""
    print(f"a={a!r}, b={b!r}")
    print(f"args={args!r}")
    print(f"limit={limit!r}")
    print(f"kwargs={kwargs!r}")


inspect_call(1, 2, 3, 4, limit=99, debug=True, mode="fast")

## Part 4 — Keyword-only parameters with bare `*`

A bare `*` in the parameter list means: **everything after this must be passed by keyword**.

This makes APIs clearer and prevents argument-order mistakes.

In [ ]:
def create_user(name, *, role, active=True):
    """role and active must be passed by keyword."""
    return {"name": name, "role": role, "active": active}


# Clear and explicit
print(create_user("Alex", role="admin", active=False))

# This would raise TypeError:
# create_user("Alex", "admin")

## Part 5 — Unpacking at call time

`*` and `**` also work when **calling** a function:

- `*` unpacks a sequence into positional arguments
- `**` unpacks a mapping into keyword arguments

This is the reverse of collection in the function definition.

In [ ]:
def add(a, b, c):
    return a + b + c


numbers = [1, 2, 3]

# Equivalent to add(1, 2, 3)
print(add(*numbers))

In [ ]:
def request(url, timeout=30, retries=3):
    return {"url": url, "timeout": timeout, "retries": retries}


defaults = {"timeout": 30, "retries": 3}
user_overrides = {"timeout": 5}

# Merge dicts: user values win over defaults
options = {**defaults, **user_overrides}

print(request("https://api.example.com/users", **options))

## Part 6 — Practical patterns (intermediate)

These patterns appear constantly in real Python code.

In [ ]:
def announce(func):
    """Minimal decorator-style wrapper using *args and **kwargs."""

    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)

    return wrapper


@announce
def multiply(a, b, factor=1):
    return a * b * factor


print(multiply(3, 4))
print(multiply(3, 4, factor=2))

In [ ]:
def log(level, message, **context):
    """Flexible logger: required fields + arbitrary context."""
    context_str = ", ".join(f"{k}={v!r}" for k, v in context.items())
    print(f"[{level}] {message} | {context_str}")


log("INFO", "User logged in", user="alex", ip="127.0.0.1")
log("ERROR", "Payment failed", order_id=42, reason="timeout")

In [ ]:
def build_query_string(**params):
    """Turn keyword params into a URL query string."""
    parts = [f"{key}={value}" for key, value in params.items()]
    return "&".join(parts)


def get(path, **params):
    """Thin API client: forward arbitrary query parameters."""
    query = build_query_string(**params)
    return f"GET {path}?{query}"


print(get("/users", page=1, limit=20, active=True))

In [ ]:
def dispatch(action, **options):
    """Choose behavior based on kwargs keys."""
    if action == "email":
        return f"Sending email to {options.get('to')}"
    if action == "sms":
        return f"Sending SMS to {options.get('phone')}"
    return "Unknown action"


print(dispatch("email", to="alex@example.com", subject="Hello"))
print(dispatch("sms", phone="+123456789", text="Code: 1234"))

## Part 7 — Common pitfalls

Avoid these mistakes when using `*args` and `**kwargs`.

In [ ]:
# Pitfall 1: wrong parameter order in definition
# def bad(a, **kwargs, *args):  # SyntaxError

# Pitfall 2: args is a tuple, not a list
def show_type(*args):
    print(type(args), args)


show_type(1, 2, 3)

# Pitfall 3: kwargs dict is local, but mutable VALUES inside it are shared
tags = ["python"]


def store(**kwargs):
    # This mutates the caller's list object, not just a local copy
    kwargs["tags"].append("saved")
    return kwargs


store(tags=tags)
print("caller tags after call:", tags)

# Safer: copy mutable values before changing them
def store_safe(**kwargs):
    data = dict(kwargs)
    data["tags"] = list(kwargs.get("tags", []))
    data["tags"].append("saved")
    return data


tags2 = ["python"]
print(store_safe(tags=tags2))
print("tags2 unchanged:", tags2)

In [ ]:
# Pitfall 4: duplicate keyword names when unpacking

def save_record(record_id, payload):
    return {"id": record_id, "payload": payload}


data = {"record_id": 1, "payload": {"name": "Alex"}}
print(save_record(**data))

# If keys do not match parameter names, TypeError occurs:
bad_data = {"id": 1, "payload": {}}
try:
    save_record(**bad_data)
except TypeError as exc:
    print("TypeError:", exc)

## Part 8 — Bridge to your learning path

| Where you saw it | Why it matters |
|------------------|----------------|
| [`Foundations/01_Python_Decorators.ipynb`](01_Python_Decorators.ipynb) | Decorator wrappers use `wrapper(*args, **kwargs)` to forward any call signature |
| [`Lessons/02_Pydantic_Graph.ipynb`](../Lessons/02_Pydantic_Graph.ipynb) | Node functions usually take explicit state, not varargs |
| Framework code (FastAPI, pytest, requests) | Helpers accept flexible options via `**kwargs` |

Once you understand collection vs unpacking, decorator wrappers become much easier to read.

In [ ]:
def trace(func):
    def wrapper(*args, **kwargs):
        print(f"args={args}, kwargs={kwargs}")
        return func(*args, **kwargs)

    return wrapper


@trace
def connect(host, port=8080, **options):
    return {"host": host, "port": port, "options": options}


connect("localhost", 443, ssl=True, timeout=5)

## Part 9 — Recap cheat sheet

### In a function definition (collect)

```python
def f(a, b, *args, limit=10, **kwargs):
    ...
```

- `a, b` — required positional parameters
- `*args` — extra positionals → `tuple`
- `limit` — keyword-only (if placed after `*`)
- `**kwargs` — extra keywords → `dict`

### At call time (unpack)

```python
f(1, 2, 3, limit=99, debug=True)
f(*[1, 2, 3], **{"limit": 99, "debug": True})
```

### When to use what

| Use | When |
|-----|------|
| Explicit parameters | The API is stable and callers should know exact fields |
| `*args` | Variable number of positional values (rare in public APIs) |
| `**kwargs` | Optional/extensible keyword options, forwarding, wrappers |
| Keyword-only (`*`) | Prevent confusing positional calls for important options |

### Common pitfalls

1. Wrong parameter order in definitions
2. Forgetting `args` is a `tuple`
3. Mutating `kwargs` in place
4. `**dict` keys must match target parameter names

**Next in `Foundations/`:** context managers (`with`), generators, or typing deep-dive.